# IMUSA — MuRIL Text Classifier (Google Colab / T4 GPU)

**Model:** `google/muril-base-cased` → 4-class Punjabi meme sentiment  
**Classes:** Motivational · Neutral · Offensive · Sarcasm  
**Metric:** Macro-F1 (primary)  
**Expected runtime:** ~20–30 min on a free T4 GPU

---

## ✅ Before running — upload ONLY these 2 files to Google Drive

Create a folder called **`IMUSA`** inside **My Drive**, then upload:

| Local file (on your PC) | Upload to (in Drive) |
|-------------------------|----------------------|
| `results/split_train.csv` | `My Drive/IMUSA/split_train.csv` |
| `results/split_val.csv`   | `My Drive/IMUSA/split_val.csv`   |

Then:  
1. **Runtime → Change runtime type → T4 GPU → Save**  
2. **Runtime → Run all** (or Ctrl+F9)

Results are saved automatically to `My Drive/IMUSA/results/text_muril/`.


In [ ]:
# Cell 1 — Verify GPU
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type > T4 GPU')


In [ ]:
# Cell 2 — Install / upgrade packages
!pip install -q transformers==4.53.2 accelerate scikit-learn


In [ ]:
# Cell 3 — Mount Google Drive and verify uploaded files
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/IMUSA')

for fname in ['split_train.csv', 'split_val.csv']:
    p = DRIVE_ROOT / fname
    tag = 'OK      ' if p.exists() else 'MISSING  <- upload this file!'
    print(f'  [{tag}]  {p}')


In [ ]:
# Cell 4 — Imports and hyperparameters
import sys, csv, json, time, random
import numpy as np
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── Hyperparameters ──────────────────────────────────────────
SEED         = 42
MODEL_NAME   = 'google/muril-base-cased'
MAX_LEN      = 128
EPOCHS       = 5
LR           = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE     = 2
DROPOUT      = 0.1

# ── Paths (Drive) ────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/IMUSA')
TRAIN_CSV  = DRIVE_ROOT / 'split_train.csv'
VAL_CSV    = DRIVE_ROOT / 'split_val.csv'
OUT_DIR    = DRIVE_ROOT / 'results' / 'text_muril'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Runtime ──────────────────────────────────────────────────
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE  = 32
NUM_WORKERS = 2
USE_AMP     = True

LABEL2ID    = {'Motivational': 0, 'Neutral': 1, 'Offensive': 2, 'Sarcasm': 3}
ID2LABEL    = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 4

print(f'Device     : {DEVICE}')
print(f'Batch size : {BATCH_SIZE}  |  AMP: {USE_AMP}')
print(f'Output     : {OUT_DIR}')


In [ ]:
# Cell 5 — Seed + Dataset class
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

def load_csv(path):
    with open(path, newline='', encoding='utf-8-sig') as fh:
        return list(csv.DictReader(fh))


class MemeTextDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row  = self.rows[idx]
        text = row.get('Text', '') or ''
        enc  = self.tokenizer(
            text, max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'image_id':       row['Id'],
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        if row.get('Category'):
            item['labels'] = torch.tensor(LABEL2ID[row['Category']], dtype=torch.long)
        return item

print('Dataset class ready.')


In [ ]:
# Cell 6 — MuRIL classifier + metrics helper
class MuRILClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden       = self.encoder.config.hidden_size  # 768
        self.drop    = nn.Dropout(dropout)
        self.head    = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out     = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls_vec = out.last_hidden_state[:, 0, :]  # [CLS]
        return self.head(self.drop(cls_vec))


def compute_metrics(preds, labels):
    return {
        'accuracy':    round(accuracy_score(labels, preds), 6),
        'macro_f1':    round(f1_score(labels, preds, average='macro',  zero_division=0), 6),
        'macro_prec':  round(precision_score(labels, preds, average='macro', zero_division=0), 6),
        'macro_rec':   round(recall_score(labels, preds, average='macro',  zero_division=0), 6),
        'per_class_f1': {
            ID2LABEL[i]: round(f1_score(labels, preds, labels=[i], average='micro', zero_division=0), 6)
            for i in range(NUM_CLASSES)
        },
        'confusion_matrix': confusion_matrix(
            labels, preds, labels=list(range(NUM_CLASSES))
        ).tolist(),
        'classification_report': classification_report(
            labels, preds,
            target_names=[ID2LABEL[i] for i in range(NUM_CLASSES)],
            zero_division=0,
        ),
    }

print('Model + metrics ready.')


In [ ]:
# Cell 7 — Training and evaluation loops
def run_epoch(model, loader, optimiser, scheduler, scaler, criterion, train):
    model.train(train)
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        tt   = batch['token_type_ids'].to(DEVICE) if 'token_type_ids' in batch else None
        lbl  = batch['labels'].to(DEVICE)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask, tt)
                loss   = criterion(logits, lbl)

        if train:
            optimiser.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimiser)
            scaler.update()
            scheduler.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(-1).cpu().numpy().tolist())
        all_labels.extend(lbl.cpu().numpy().tolist())

    return total_loss / len(loader), compute_metrics(all_preds, all_labels)


@torch.no_grad()
def run_eval(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels, all_probs, all_ids = 0.0, [], [], [], []

    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        tt   = batch['token_type_ids'].to(DEVICE) if 'token_type_ids' in batch else None
        lbl  = batch['labels'].to(DEVICE)

        logits = model(ids, mask, tt)
        total_loss += criterion(logits, lbl).item()

        probs = torch.softmax(logits, -1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(probs.argmax(-1).tolist())
        all_labels.extend(lbl.cpu().numpy().tolist())
        all_ids.extend(batch['image_id'])

    return total_loss / len(loader), compute_metrics(all_preds, all_labels), \
           all_preds, all_labels, all_probs, all_ids

print('Loops ready.')


In [ ]:
# Cell 8 — Build loaders, model, optimiser
print('Loading tokenizer ...')
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

train_rows = load_csv(TRAIN_CSV)
val_rows   = load_csv(VAL_CSV)
print(f'Train: {len(train_rows)}  |  Val: {len(val_rows)}')

train_loader = DataLoader(
    MemeTextDataset(train_rows, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    MemeTextDataset(val_rows, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

print('Loading MuRIL ...')
model     = MuRILClassifier(MODEL_NAME, NUM_CLASSES, DROPOUT).to(DEVICE)
n_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')

# Weighted CrossEntropyLoss (handles Offensive class imbalance)
label_counts = [0] * NUM_CLASSES
for r in train_rows:
    label_counts[LABEL2ID[r['Category']]] += 1
tot     = sum(label_counts)
weights = torch.tensor(
    [tot / (NUM_CLASSES * c) for c in label_counts], dtype=torch.float32
).to(DEVICE)
print('Class weights:', {ID2LABEL[i]: round(weights[i].item(), 3) for i in range(NUM_CLASSES)})
criterion = nn.CrossEntropyLoss(weight=weights)

no_decay  = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
optimiser = AdamW([
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
], lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimiser, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)
scaler = torch.amp.GradScaler('cuda')
print('Ready to train.')


In [ ]:
# Cell 9 — Training loop with early stopping on val macro-F1
best_f1, patience_ctr, log_rows = -1.0, 0, []
best_ckpt = OUT_DIR / 'best_model'
best_vp = best_vl = best_vprob = best_vids = best_metrics = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    print(f'\n=== Epoch {epoch}/{EPOCHS} ===')

    train_loss, train_m = run_epoch(
        model, train_loader, optimiser, scheduler, scaler, criterion, train=True
    )
    val_loss, val_m, vp, vl, vprob, vids = run_eval(model, val_loader, criterion)
    elapsed = time.time() - t0

    print(f'  train  loss={train_loss:.4f}  macro-F1={train_m["macro_f1"]:.4f}  acc={train_m["accuracy"]:.4f}')
    print(f'  val    loss={val_loss:.4f}  macro-F1={val_m["macro_f1"]:.4f}  acc={val_m["accuracy"]:.4f}  ({elapsed:.0f}s)')
    print(f'  val per-class F1: {val_m["per_class_f1"]}')

    log_rows.append({
        'epoch': epoch,
        'train_loss': round(train_loss, 6), 'train_macro_f1': train_m['macro_f1'], 'train_acc': train_m['accuracy'],
        'val_loss':   round(val_loss,   6), 'val_macro_f1':   val_m['macro_f1'],   'val_acc':   val_m['accuracy'],
        'val_macro_prec': val_m['macro_prec'], 'val_macro_rec': val_m['macro_rec'],
        'elapsed_s': round(elapsed, 1),
    })

    if val_m['macro_f1'] > best_f1:
        best_f1 = val_m['macro_f1']
        patience_ctr = 0
        best_vp, best_vl, best_vprob, best_vids, best_metrics = vp, vl, vprob, vids, val_m
        best_ckpt.mkdir(parents=True, exist_ok=True)
        model.encoder.save_pretrained(best_ckpt)
        tokenizer.save_pretrained(best_ckpt)
        torch.save(model.head.state_dict(), best_ckpt / 'classifier_head.pt')
        print(f'  *** New best val macro-F1 = {best_f1:.4f} — checkpoint saved ***')
    else:
        patience_ctr += 1
        print(f'  No improvement. Patience {patience_ctr}/{PATIENCE}')
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

print('Training complete.')


In [ ]:
# Cell 10 — Save training log, predictions, metrics to Drive

# Training log CSV
with open(OUT_DIR / 'training_log.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=list(log_rows[0].keys()))
    w.writeheader(); w.writerows(log_rows)
print('Saved: training_log.csv')

# Val predictions + probabilities
prob_cols = [f'prob_{ID2LABEL[i]}' for i in range(NUM_CLASSES)]
with open(OUT_DIR / 'val_predictions.csv', 'w', newline='', encoding='utf-8-sig') as fh:
    w = csv.DictWriter(fh, fieldnames=['Id', 'TrueLabel', 'PredLabel', 'Correct'] + prob_cols)
    w.writeheader()
    for i, iid in enumerate(best_vids):
        tl = ID2LABEL[best_vl[i]]; pl = ID2LABEL[best_vp[i]]
        row = {'Id': iid, 'TrueLabel': tl, 'PredLabel': pl, 'Correct': int(tl == pl)}
        for j, pc in enumerate(prob_cols):
            row[pc] = round(float(best_vprob[i][j]), 6)
        w.writerow(row)
print('Saved: val_predictions.csv')

# Metrics JSON
best_epoch  = next(r['epoch'] for r in log_rows if r['val_macro_f1'] == best_f1)
metrics_out = {
    'model': MODEL_NAME, 'seed': SEED, 'best_epoch': best_epoch,
    'best_val_macro_f1':  best_metrics['macro_f1'],
    'accuracy':           best_metrics['accuracy'],
    'macro_precision':    best_metrics['macro_prec'],
    'macro_recall':       best_metrics['macro_rec'],
    'per_class_f1':       best_metrics['per_class_f1'],
    'confusion_matrix':   best_metrics['confusion_matrix'],
    'label_order':        [ID2LABEL[i] for i in range(NUM_CLASSES)],
    'classification_report': best_metrics['classification_report'],
    'training_log':       log_rows,
    'generated_at':       datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}
with open(OUT_DIR / 'metrics.json', 'w', encoding='utf-8') as fh:
    json.dump(metrics_out, fh, ensure_ascii=False, indent=2)
print('Saved: metrics.json')

print(f'\nAll results saved to: {OUT_DIR}')


In [ ]:
# Cell 11 — Final results summary
print('=' * 60)
print(f'  BEST RESULTS  (epoch {metrics_out["best_epoch"]})')
print('=' * 60)
print(f'  Accuracy      : {metrics_out["accuracy"]:.4f}')
print(f'  Macro-F1      : {metrics_out["best_val_macro_f1"]:.4f}')
print(f'  Macro-Prec    : {metrics_out["macro_precision"]:.4f}')
print(f'  Macro-Recall  : {metrics_out["macro_recall"]:.4f}')
print()
print('  Per-class F1:')
for cls, val in metrics_out['per_class_f1'].items():
    print(f'    {cls:<15} {val:.4f}')
print()
print('  Confusion matrix (rows=true, cols=pred):')
lbls = metrics_out['label_order']
print('    ' + '  '.join(f'{l[:6]:>8}' for l in lbls))
for i, row in enumerate(metrics_out['confusion_matrix']):
    print(f'  {lbls[i]:<14}' + '  '.join(f'{v:>8}' for v in row))
print()
print(metrics_out['classification_report'])


In [ ]:
# Cell 12 (Optional) — Download results ZIP to your PC
# Skip this if you prefer to access files directly from Drive
import shutil
zip_path = '/content/muril_text_results.zip'
shutil.make_archive('/content/muril_text_results', 'zip', str(OUT_DIR))
from google.colab import files
files.download(zip_path)
print('Download started — check your browser Downloads folder.')
